In [ ]:
!pip install -r requirements.txt
!pip install transformers shap lime pytorch-gradcam

In [ ]:
import os
import shutil
from google.colab import drive

# 1. Mount Drive (Essential for accessing your saved data.zip)
drive.mount('/content/drive')

# 2. Define your source and destination
# Use the exact path we used for the download earlier
SOURCE_ZIP = "/content/drive/MyDrive/Colab Notebooks/data/data.zip"
DESTINATION_DIR = "/content/dataset"

# 3. Create the local directory
os.makedirs(DESTINATION_DIR, exist_ok=True)

print("Starting Transfer from Drive to Local SSD (this is fast)...")

# 4. Copy the zip file (using !cp for high-speed Linux transfer)
!cp "{SOURCE_ZIP}" /content/data_local.zip

print("Transfer complete. Starting extraction (this may take 15-20 mins)...")

# 5. Unzip the file into the local folder
# -q flag (quiet) is CRITICAL here; printing 112,000 filenames will crash your VS Code tab.
!unzip -q /content/data_local.zip -d "{DESTINATION_DIR}"

# 6. Clean up the local zip to save space (now that it's extracted)
os.remove("/content/data_local.zip")

print(f"Done! Your {len(os.listdir(DESTINATION_DIR))} images are ready at {DESTINATION_DIR}")

In [ ]:
%%writefile src/config.py
import os

ROOT_DIR = os.path.abspath(os.path.join(os.path.dirname(__file__), ".."))

DATASET_DIR = "/content/dataset"
CSV_PATH = os.path.join(DATASET_DIR, "Data_Entry_2017.csv")

IMAGE_SIZE = 384
NUM_CLASSES = 14
BATCH_SIZE = 8  # if OOM on GPU, lower this value (e.g. 4).
EPOCHS = 10
LR = 1e-4

MODEL_NAME = "radjepa"

In [ ]:
!python src/main.py

In [ ]:
import torch
import os
from src.model import get_model
from src.config import MODEL_NAME

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = get_model()

# Define the path to your saved checkpoint (adjust epoch number if needed)
checkpoint_path = os.path.join("checkpoints", f"{MODEL_NAME}_epoch_10.pth")

model.load_state_dict(torch.load(checkpoint_path, map_location=device, weights_only=True))
model.to(device)

model.eval()

print(f"Successfully loaded {MODEL_NAME} weights from {checkpoint_path}")
print("Model is ready for inference or testing!")